In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy


from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/"
data = sc.read_h5ad(data_annotated_path + "joint_annotated.h5ad")
# data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden.h5ad")

data.shape

In [ ]:
# CHECK POPULATION LOSS DUMBASS
# RODS CONES 
# METHODS 
# GSEA ON THE DIFF 

# hey doof, it's not 10, it's 3 lol. https://markergenes.com/singleWeighted

# Ah shit, wait, RGC = retinal ganglion or RGC = radial glia?

# Building A Timeline

In [ ]:
plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [data.obs['log_total_counts'][data.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()


# PCA IQR Timeline, Big Sheet

We can briefly look at the timecourse of each individual PC across the entire dataset with no filtering to sanity check our data and see the overall level of stability. We also want to observe whether or not there are obvious outliers

In [ ]:
def extract_iqrs(data,low_percentile=25,high_percentile=75):
    return {
        'low':np.percentile(data,low_percentile),
        'center':np.median(data),
        'high':np.percentile(data,high_percentile),
    }

def extract_sems(data,sem_multiple=2):
    sem = scipy.stats.sem(data)
    mean = np.mean(data)
    return {
        'low':mean - (sem*sem_multiple),
        'center':mean,
        'high':mean + (sem*sem_multiple),
    }

def filled_centerline_to_ax(
    ax,#times,
    shadowed_segments,
    label=None,fill_label=None,ax_label=None,
    **kwargs
):

    centerline_to_ax(ax,shadowed_segments,label=label,ax_label=ax_label)

    times = sorted(shadowed_segments.keys())
    ax.fill_between(
        times,
        [shadowed_segments[t]['low'] for t in times],
        [shadowed_segments[t]['high'] for t in times],
        label=fill_label,alpha=.2
    )

    if fill_label:
        ax.legend()
    return ax

def centerline_to_ax(
    ax,
    shadowed_segments,
    label=None,ax_label=None,
    **kwargs
):
    times = sorted(shadowed_segments.keys())

    ax.plot(
        times,
        [shadowed_segments[t]['center'] for t in times],
        label=label,
        **kwargs
    )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel(ax_label)
    ax.set_title(ax_label)
    if label:
        ax.legend()
    return ax

def get_compound_masks(
    data
):
    times = sorted(set(data.obs['time']))
    time_masks = [
        data.obs['time'] == t
        for t in times
    ]

    control_mask =  data.obs['exp_condition'] == "Cntr"
    injury_mask = data.obs['exp_condition'] == "Mtz"

    combined_masks = {
        'control':{
            time:control_mask & time_mask for time,time_mask in zip(times,time_masks)
        },
        'injury':{
            time: injury_mask & time_mask for time,time_mask in zip(times,time_masks)
        }
    }

    return combined_masks

def map_over_leaves(compound,fn):
   return {
    outer_key: {inner_key: fn(v) for inner_key, v in inner.items()} 
    for outer_key, inner in compound.items()
} 

def get_compound_scores(
    compound_masks,target
):
    return map_over_leaves(compound_masks,lambda mask: target[mask])

def get_compound_iqrs(
    compound_scores
):
    return map_over_leaves(compound_scores, extract_iqrs)

def get_compound_means(
    compound_scores
):
    return map_over_leaves(compound_scores, extract_sems)


def plot_timecourse_from_compound(
    compound_iqrs,ax,
    label=None,fill_label=None,ax_label=None,plot_shadow=True,
):
    for condition in compound_iqrs:

        if plot_shadow:
            filled_centerline_to_ax(
                ax,compound_iqrs[condition],
                label=f"{label} : {condition}",
                fill_label=fill_label,
                ax_label=ax_label
            )
        else:
            centerline_to_ax(
                ax,compound_iqrs[condition],
                label=f"{label} : {condition}",
                ax_label=ax_label                
            )

def plot_timecourse(
    target,compound_masks,ax,
    label=None,fill_label=None,ax_label=None,plot_shadow=True,central_tendency="median"
):
    compound_scores = get_compound_scores(compound_masks,target)
    compound_shadow = None
    shadow_label = None
    match central_tendency:
        case "median": 
            compound_shadow = get_compound_iqrs(compound_scores)
            shadow_label = "IQR"
        case "mean": 
            compound_shadow = get_compound_means(compound_scores)
            shadow_label = "SEM"
        case _: raise ValueError(f"Invalid central tendency: {central_tendency}")
    plot_timecourse_from_compound(
        compound_shadow,ax,
        # compound_means,ax,
        fill_label=shadow_label,label=label,
        ax_label=ax_label,plot_shadow=plot_shadow
    )

In [ ]:
compound_masks = get_compound_masks(data)

fig,axes = plt.subplots(5,10,figsize=(30,15))
for i in range(50):
    ax = axes[i//10,i%10]

    target = data.obsm['X_pca'][:,i]
    plot_timecourse(
        target,compound_masks,ax,
        ax_label=f'PC{i}',fill_label='IQR'
    )
fig.tight_layout()
fig.show()

# control_mask =  data.obs['exp_condition'] == "Cntr"
# injury_mask = data.obs['exp_condition'] == "Mtz"

# times = [12,24,48,72]

# time_masks = [
#     data.obs['time'] == t
#     for t in times
# ]

# fig,axes = plt.subplots(5,10,figsize=(30,15))
# for i in range(50):

#     time_control_iqrs = {}
#     time_mutant_iqrs = {}
#     for t,time_mask in zip(times,time_masks):
#         combined_control_mask = control_mask & time_mask
#         combined_injury_mask = injury_mask & time_mask

#         control_target = data.obsm['X_pca'][:,i][combined_control_mask]
#         mutant_target = data.obsm['X_pca'][:,i][combined_injury_mask]

#         time_control_iqrs[t] = extract_iqrs(control_target)
#         time_mutant_iqrs[t] = extract_iqrs(mutant_target)
    
#     ax = axes[i//10,i%10]

#     filled_centerline_to_ax(
#         ax,times,time_control_iqrs,
#         # label="control",fill_label="control (IQR)",
#         ax_label=f"PC{i+1}",
#     )
#     filled_centerline_to_ax(
#         ax,times,time_mutant_iqrs,
#         #label="mutant",fill_label="mutant (IQR)",
#         ax_label=f"PC{i+1}",
#     )

# fig.tight_layout()
# fig.show()

# Basic DESeq

In [ ]:
data.layers['raw']

In [ ]:
data.layers['raw'] = np.array(data.layers['raw'].astype(int).todense())

In [ ]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=data.layers['raw'],
    metadata=data.obs,
    design="~exp_condition",
    refit_cooks=True,
    inference=inference,
)

In [ ]:
dds.deseq2()

In [ ]:
ds = DeseqStats(dds, contrast=["condition", "B", "A"], inference=inference)

# Cell Death Scoring

The particulars of producing these scores are in the sister file "annotations", but this is the set of gene sets used by the TBI guys. They can be updated with alternatives on demand. (Swap in the gene sets and map the homologs to zebrafish genome)

We want to plot the timecourse of the score, as well as the scores over the UMAP to see if there are any obvious patterns.

Brief note: representing an uncertainty score here is a little fraught, I'm not a huge fan of either the IQR or the SEM, the former is too vague and the latter is too conservative (as written it is treating each cell as a unique observation)

I want to check if there are biological replicate annotations, if so we can re-calculate the SEM on a per-sample basis, which will better represent the actual variation certainty. Please stand by for this.


In [ ]:
# necroptosis_translated_signature = ['pgam5', 'ripk3', 'dnm1l', 'ripk1l']
# data.var['necroptosis'] = data.var_names.isin(necroptosis_translated_signature)
# sc.tl.score_genes(data,data.var_names[data.var['parthanatos']],score_name="parthanatos")

In [ ]:
sc.tl.score_genes(data,data.var_names[data.var['parthanatos']],score_name="parthanatos")
sc.tl.score_genes(data,data.var_names[data.var['necroptosis']],score_name="necroptosis")
sc.tl.score_genes(data,data.var_names[data.var['apoptosis']],score_name="apoptosis")



### Sham Signature 

In [ ]:
# Let's also construct a sham gene signature to get a sense for "baseline" signal

# First we want to establish the overall number of genes in ext signatures:

print((
    f'Parthanatos: {np.sum(data.var['parthanatos'])}\n'
    f'Necroptosis: {np.sum(data.var['necroptosis'])}\n'
    f'Apoptosis: {np.sum(data.var['apoptosis'])}\n'
))

In [ ]:
n_features = len(data.var_names)
n_shams = 5
small_sham_signature_masks = [np.random.randint(0,n_features,size=10) for _ in range(n_shams)]
medium_sham_signature_masks = [np.random.randint(0,n_features,size=17) for _ in range(n_shams)]
large_sham_signature_masks = [np.random.randint(0,n_features,size=24) for _ in range(n_shams)]

for i,sham in enumerate(small_sham_signature_masks + medium_sham_signature_masks + large_sham_signature_masks):
    sc.tl.score_genes(data,data.var_names[sham],score_name=f'sham_{i}')
    



In [ ]:
data.obs = data.obs.copy()

In [ ]:
# Let's plot a large number sans shadows actually. Just simple overlay
compound_masks = get_compound_masks(data)

sham_timecourses = []

# fig,axes = plt.subplots(2,3,figsize=(10,6))
for sham in small_sham_signature_masks:
    sc.tl.score_genes(data,data.var_names[sham],score_name=f'sham_tmp')
    sham_timecourses.append(data.obs['sham_tmp'])
    # target = data.obs['sham_tmp']
    # plot_timecourse(
    #     target,compound_masks=compound_masks,ax=axes[0,0],)
# for sham in medium_sham_signature_masks:
#     pass
# fig.tight_layout()
# fig.show()

In [ ]:
compound_masks = get_compound_masks(data)

fig,axes = plt.subplots(3,5,figsize=(20,10))
for i,ax in enumerate(axes.flatten()):
    target = data.obs[f'sham_{i}']
    plot_timecourse(
        target,compound_masks,ax,ax_label=f'Sham Signature {i}',
        central_tendency="mean"
    )
fig.tight_layout()
fig.show()

# Timecourse Plot

In [ ]:
# Let's plot a timecourse for each between injury and ctrl 
control_mask =  data.obs['exp_condition'] == "Cntr"
injury_mask = data.obs['exp_condition'] == "Mtz"

times = [12,24,48,72]

time_masks = [
    data.obs['time'] == t
    for t in times
]

# Timecourse with an IQR shadow

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    # We build the IQR shadows here as dictionaries

    time_control_iqrs = {}
    time_injury_iqrs = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_injury_mask = injury_mask & time_mask

        control_data = data.obs[score][combined_control_mask]
        injury_data = data.obs[score][combined_injury_mask]

        time_control_iqrs[t] = extract_iqrs(control_data)
        time_injury_iqrs[t] = extract_iqrs(injury_data)

    # Actual plotting
    ax = axes[i]
    filled_centerline_to_ax(
        ax,time_control_iqrs,
        label="control (median)",fill_label="control (IQR)",ax_label=f"{score}, (all cells)",
    )
    filled_centerline_to_ax(
        ax,time_injury_iqrs,
        label="injury (median)",fill_label="injury (IQR)",ax_label=f"{score}, (all cells)",
    )
    ax.legend(loc="upper left")

fig.tight_layout()
fig.show()

# Timecourse with an SEM 

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    # Build SEM shadows

    time_control_sem_range = {}
    time_injury_sem_range = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_injury_mask = injury_mask & time_mask

        control_target = data.obs[score][combined_control_mask]
        injury_target = data.obs[score][combined_injury_mask]

        time_control_sem_range[t] = extract_sems(control_target)
        time_injury_sem_range[t] = extract_sems(injury_target)

    # Actual plotting
    ax = axes[i]
    filled_centerline_to_ax(
        ax,time_control_sem_range,
        label="control (mean)",fill_label="control (SEM)",ax_label=f"{score}, (all cells)",
    )
    filled_centerline_to_ax(
        ax,time_injury_sem_range,
        label="injury (mean)",fill_label="injury (SEM)",ax_label=f"{score}, (all cells)",
    )
    ax.legend(loc="best")

fig.tight_layout()
fig.show()

### UMAP Score Plots

We can see that the distribution of the scores is similar but not fully overlapping. We can probably confirm this intuition with a bunch of pairwise scatters, but this basically tracks with the lit anyway afaik. 

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(25,5))
axes = axes.flatten()

sc.pl.umap(data,color='parthanatos',ax=axes[0],show=False)
sc.pl.umap(data,color='necroptosis',ax=axes[1],show=False)
sc.pl.umap(data,color='apoptosis',ax=axes[2],show=False)

fig.show()

# Leiden 

We will use Leiden to do unsupervised partitioning and then look at the partitions to see how well they match up with already-known markers. This will let us easily pluck out eg RGCs without being too bothered about re-rolling a classifier. 

### Sundries

In [ ]:
sc.tl.leiden(data, resolution=1)

In [ ]:
sc.pl.umap(data, color=['leiden'],legend_loc='on data')

In [ ]:
sc.tl.rank_genes_groups(data, 'leiden')
sc.pl.rank_genes_groups(data, n_genes=25)


In [ ]:
# top_3_across_all.flatten()
# sc.pl.umap(data,color='pax6b')

# [(g,g in data.var_names) for g in ['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',]]
# [(g,g in data.var_names) for g in ['gnb3a','gng13b',]]
# [g for g in data.var_names if 'gnb' in g]


In [ ]:
population_marker_sets = {
    'cones': ['gnat2','arr3a','arr3b'],
    'rods': ['rho','gnat1','gngt1','sagb','pde6a','pde6b','nrl'],
    'muller_glia':['rlbp1a','rx1','crabp1a','gpr37a',],
    'rgcs':['rbpms2b','isl2b','uchl1','rbpms2a','pou4f2'],
    'rgc_secondary':['nrn1a','isl2b','islr2',],
    'microglia':['mfap4','mpeg1.1','csf1ra','spi1b'],
    'bipolar_cells':['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',], # It looks like the ENTIRETY of bipolar cells got filtered?
    'amacrine_cells':['pax6b','slc6a1b','pax10','nhlh2','pvalb6'], # pax6a got filtered
    'horizontal_cells':['onecutl','tnr','rem1','CR361564.1','cx52.9','ompa','opn9'], # onecut ambiguous, cx55.5, slc4a5b seem fully missing from the index
    'undifferentiated':['sox2','olig2','hes6','hmgn2','hmga1a','atoh7'],
    'pr_precursors':['tulp1a','opn6a','tmem244']
}
flat_population_markers = [g for l in population_marker_sets.values() for g in l]
flat_population_markers

### Plot

In [ ]:
# sc.pl.dotplot(data, ['rbpms2a','rbpms2b','nrn1a','isl2b','islr2','frem2a','ptn','sox2'],groupby="leiden",figsize=(20,10))
# sc.pl.dotplot(data,flat_population_markers,groupby="leiden",figsize=(10,10))
sc.pl.dotplot(data,population_marker_sets,groupby="leiden",figsize=(10,8))
# sc.pl.heatmap(data,var_names=flat_population_markers,groupby="leiden",swap_axes=True,figsize=(30,10))

Leiden is stochastic so we have run this once and saved it for posterity. To be clear: the partitions Leiden creates are highly stable, but they are discovered in random order, so there is always roughly the same population of cells identified as a cluster, but sometimes it is cluster 5 and sometimes it is cluster 15.

If you wish to re-run this analysis you need only use the marker gene dot plot to re-annotate the target leiden clusters that identify RGC subpops. This logic is in the next section (labeled RGC mask)

In [ ]:
mapping = {
    k:k for k in data.obs['leiden'].unique()
}

mapping |= {
    '0': 'Bipolar cells small',
    '1': 'Bipolar cells none',
    '2': 'Bipolar cells big',
    '3': 'Bipolar cells big',
    '4': 'Amacrine cells A',
    '5': 'Bipolar cells small',
    '6': 'Cones',
    '9': 'Horizontal cells',
    '10': 'Muller glia',
    '11': 'Horizontal cells',
    '12': 'Cones',
    '13': 'Bipolar cells none',
    '14': 'Bipolar cells small',
    '16': 'Amacrine cells B',
    '17': 'Bipolar cells small',
    
    '19': 'Cones',
    '20': 'Amacrine cells C',
    '21': 'Bipolar cells none',
    '22': 'Bipolar cells small',
    '25': 'RGCs',
    '26': 'Rods',
    '30': 'Microglia',
    '32': 'Horizontal cells',
    '33': 'trash'


    # '8': 'Muller glia',
    # '11': 'RGCs',
    # '21': 'Rods',
    # '9': 'Cones',
    # '16':'Cones',
    # '5':'Cones',
    # '6':'Horizontal cells',
    # '7': 'Horizontal cells',
    # '4': 'Amacrine cells',
    # '14': 'Amacrine cells',
    # '19': 'Amacrine cells',
    # '26': 'Microglia',
}
# mapping
# Create a new column with renamed clusters
data.obs['leiden'] = data.obs['leiden'].map(mapping).astype('category')


In [ ]:
sc.pl.umap(data,color='leiden',legend_loc='on data')

In [ ]:
# Commented out to avoid over-writing the frozen analysis
# sc.write("full_annotations_leiden.h5ad",data)

# Subset to Radial Glia

We want to check if the behavior of RGCs in particular is different than the rest of the pop with respect to the scores of interest.

We will isolate just the RGCs using leiden mapping, and then re-plot some of our previous analysis

In [ ]:
# Edit this if the leiden mappings shift because the analysis was re-run
# rgc_mask_both = (data.obs['leiden'] == '3') | (data.obs['leiden'] == '10')
# rgc_mask_10 = data.obs['leiden'] == '10'
# rgc_mask_3 = data.obs['leiden'] == '3'
# rgc_mask_19 = data.obs['leiden'] == '19'
# rgc_both = data[rgc_mask_both]
# rgc_10 = data[rgc_mask_10]
# rgc_3 = data[rgc_mask_3]
# non_rgc = data[~rgc_mask_both]
# rgc = rgc_both

# muller_glia_mask = data.obs['leiden'] == "8"
# muller_glia = data[muller_glia_mask]
# non_muller_glia = data[~muller_glia_mask]

# rgc_mask = data.obs['leiden'] == '11'
# rgc = data[rgc_mask]
# non_rgc = data[~rgc_mask]

# microglia_mask = data.obs['leiden'] == "27"
# non_microglia = data[~microglia_mask]
# microglia = data[microglia_mask]

# amacrine_mask = data.obs['leiden'] == "19"
# non_amacrine = data[~amacrine_mask]
# amacrine = data[amacrine_mask]

# mask_14 = data.obs['leiden'] == "14"
# non_mask_14 = data[~mask_14]
# data_14 = data[mask_14]

# mask_16 = data.obs['leiden'] == "16"
# non_mask_16 = data[~mask_16]
# data_16 = data[mask_16]

# mask_18 = data.obs['leiden'] == "18"
# non_mask_18 = data[~mask_18]
# data_18 = data[mask_18]



In [ ]:
# These timecourses are focused in on RGCs. 
# Second verse same as first

# cell_data = {
#     "rgc":rgc,
#     "non_rgc":non_rgc,
#     "muller_glia":muller_glia,
#     "non_muller_glia":non_muller_glia,
#     "microglia":microglia,
#     "non_microglia":non_microglia,
#     "amacrine":amacrine,
#     "non_amacrine":non_amacrine,
#     "data_14":data_14,
#     "non_data_14":non_mask_14,
#     "data_16":data_16,
#     "non_data_16":non_mask_16,
#     "data_18":data_18,
#     "non_data_18":non_mask_18
# }

# fig,axes = plt.subplots(14,3,figsize=(15,45))
# for i,cell_type in enumerate(cell_data.keys()):
#     compound_masks = get_compound_masks(cell_data[cell_type])
#     for j,score in enumerate(['parthanatos','necroptosis','apoptosis']):
#         ax = axes[i][j]
#         target_scores = cell_data[cell_type].obs[score]
#         plot_timecourse(
#             target_scores,compound_masks,ax,central_tendency="mean"
#         )

#         ax.set_title(f"{score} ({cell_type})")
# fig.tight_layout()
# fig.show()


# times = [12,24,48,72]

# time_rgc_masks = [
#     rgc.obs['time'] == t
#     for t in times
# ]

# control_rgc_mask =  rgc.obs['exp_condition'] == "Cntr"
# injury_rgc_mask = rgc.obs['exp_condition'] == "Mtz"

# fig,axes = plt.subplots(1,3,figsize=(15,5))
# for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

#     time_control_rgc_iqrs = {}
#     time_injury_rgc_iqrs = {}

#     for t,time_rgc_mask in zip(times,time_rgc_masks):

#         combined_rgc_control_mask = control_rgc_mask & time_rgc_mask
#         combined_rgc_injury_mask = injury_rgc_mask & time_rgc_mask

#         time_control_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_control_mask])
#         time_injury_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_injury_mask])

#     ax = axes[i]
    
#     filled_centerline_to_ax(
#         ax,times,time_control_iqrs,
#         label="control (median)",fill_label="control (IQR)",ax_label=f"{score}, (RGC)",
#     )
#     filled_centerline_to_ax(
#         ax,times,time_injury_iqrs,
#         label="injury (median)",fill_label="injury (IQR)",ax_label=f"{score}, (RGC)",
#     )

# fig.tight_layout()
# fig.show()

# fig,axes = plt.subplots(1,3,figsize=(15,5))
# for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

#     time_control_rgc_sem_range = {}
#     time_injury_rgc_sem_range = {}

#     for t,time_rgc_mask in zip(times,time_rgc_masks):

#         combined_rgc_control_mask = control_rgc_mask & time_rgc_mask
#         combined_rgc_injury_mask = injury_rgc_mask & time_rgc_mask

#         time_control_sem_range[t] = extract_sems(rgc.obs[score][combined_rgc_control_mask])
#         time_injury_sem_range[t] = extract_sems(rgc.obs[score][combined_rgc_injury_mask])

#     ax = axes[i]
    
#     filled_centerline_to_ax(
#         ax,times,time_control_sem_range,
#         label="control (mean)",fill_label="control (SEM)",ax_label=f"{score} (RGC)",
#     )
#     filled_centerline_to_ax(
#         ax,times,time_injury_sem_range,
#         label="injury (mean)",fill_label="injury (SEM)",ax_label=f"{score} (RGC)",
#     )

# fig.tight_layout()
# fig.show()


# n_leiden = len(data.obs['leiden'].unique())
# for i in range(n_leiden):

def plot_origin_axes(ax,plot_x=True,plot_y=True):
    xmin,xmax = ax.get_xlim()
    # xmin = min(0.,xmin)
    ymin,ymax = ax.get_ylim()
    # ymin = min(0.,ymin)
    if plot_x:
        ax.plot([xmin,xmax],[0,0],'k--')
    if plot_y:
        ax.plot([0,0],[ymin,ymax],'k--')
    return ax

ylims = {
    'parthanatos':(-.05,.1),
    'necroptosis':(-.05,.1),
    'apoptosis':(-.05,.1),
}

all_leiden = sorted(data.obs['leiden'].unique())
all_leiden.remove('trash')

for leiden in all_leiden:

    leiden_data = data[data.obs['leiden'] == str(leiden)]
    non_leiden_data = data[data.obs['leiden'] != str(leiden)]

    compound_masks = get_compound_masks(leiden_data)
    non_compound_masks = get_compound_masks(non_leiden_data)
    # fig,axes = plt.subplots(2,3,figsize=(10,7))
    fig,axes = plt.subplots(1,3,figsize=(10,5))

    fig.suptitle(f"Leiden {leiden}")
    for j,score in enumerate(['parthanatos','necroptosis','apoptosis']):
        # ax = axes[0][j]
        ax = axes[j]
        target_scores = leiden_data.obs[score]
        plot_timecourse(
            target_scores,compound_masks,ax,central_tendency="mean",plot_shadow=False,
            label=leiden
        )
        ax.set_title(f"{score} ({leiden})")
        ax.set_ylim(*ylims[score])

        # ax = axes[1][j]
        target_scores = non_leiden_data.obs[score]
        plot_timecourse(
            target_scores,non_compound_masks,ax,central_tendency="mean",plot_shadow=False,
            label="Rest"
        )
        ax.set_title(f"{score}")
        ax.set_ylim(*ylims[score])

        plot_origin_axes(ax,plot_x=True,plot_y=False)

    fig.tight_layout()
    fig.show()


# RGC Vs Rest Comparison

Let's compare the RGCs to the rest of the population to observe any differences for our scores of interest

In [ ]:
# non_rgc = data[~rgc_mask]

times = [12,24,48,72]

rgc_time_masks = [
    rgc.obs['time'] == t
    for t in times
]

non_rgc_time_masks = [
    non_rgc.obs['time'] == t
    for t in times
]

rgc_control_mask =  rgc.obs['exp_condition'] == "Cntr"
rgc_injury_mask = rgc.obs['exp_condition'] == "Mtz"

non_rgc_control_mask =  non_rgc.obs['exp_condition'] == "Cntr"
non_rgc_injury_mask = non_rgc.obs['exp_condition'] == "Mtz"

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_rgc_control_iqrs = {}
    time_rgc_injury_iqrs = {}
    time_non_rgc_control_iqrs = {}
    time_non_rgc_injury_iqrs = {}

    for t,(rgc_time_mask,non_rgc_time_mask) in zip(times,zip(rgc_time_masks,non_rgc_time_masks)):

        combined_rgc_control_mask = rgc_control_mask & rgc_time_mask
        combined_rgc_injury_mask = rgc_injury_mask & rgc_time_mask

        combined_non_rgc_control_mask = non_rgc_control_mask & non_rgc_time_mask
        combined_non_rgc_injury_mask = non_rgc_injury_mask & non_rgc_time_mask

        time_rgc_control_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_control_mask])
        time_rgc_injury_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_injury_mask])

        time_non_rgc_control_iqrs[t] = extract_iqrs(non_rgc.obs[score][combined_non_rgc_control_mask])
        time_non_rgc_injury_iqrs[t] = extract_iqrs(non_rgc.obs[score][combined_non_rgc_injury_mask])

    ax = axes[i]

    # We're going to discard all the IQR stuff here because the shadows make this more or less unreadable. 
    # But I do think it's important to visualize this in overlay to convey the idea

    ax.plot(times, [time_rgc_control_iqrs[t]['center'] for t in times], label="RGC Control", color='r')
    ax.plot(times, [time_rgc_injury_iqrs[t]['center'] for t in times], label="RGC injury", color='b')
    ax.plot(times, [time_non_rgc_control_iqrs[t]['center'] for t in times], label="Non-RGC Control", color='r', linestyle='--',alpha=.3)
    ax.plot(times, [time_non_rgc_injury_iqrs[t]['center'] for t in times], label="Non-RGC injury", color='b', linestyle='--',alpha=.3)

    ax.set_xlabel("Time (h)")
    ax.set_ylabel(f"{score}")
    ax.set_title(f"{score} (RGC vs Non-RGC)")
    ax.legend()

fig.tight_layout()
fig.show()


# PCA sanity check

We should check if any computed PCs meaningfully correspond to the computed cell death scores, as this would be a simple way to verify whether or not this is a relatively orthogonal signal

In [ ]:
corr = np.corrcoef(data.obsm['X_pca'].T,data.obs[['parthanatos','necroptosis','apoptosis']].T)[50:,:50]

plt.figure()
plt.imshow(
    corr.T,
    cmap='bwr',vmin=-1,vmax=1,
    aspect='auto'
)
plt.ylabel("PCs")
plt.xticks(np.arange(3),labels=['parthanatos','necroptosis','apoptosis'])
plt.colorbar(label="Pearson Correlation")
plt.title("Cell Death Scores vs PCA Scores")
plt.show()

In [ ]:
parthanatos_min,parthanatos_max = np.min(corr[0]),np.max(corr[0])
necroptosis_min,necroptosis_max = np.min(corr[1]),np.max(corr[1])
apoptosis_min,apoptosis_max = np.min(corr[2]),np.max(corr[2])

parthanatos_min,parthanatos_max = np.around(parthanatos_min,3),np.around(parthanatos_max,3)
necroptosis_min,necroptosis_max = np.around(necroptosis_min,3),np.around(necroptosis_max,3)
apoptosis_min,apoptosis_max = np.around(apoptosis_min,3),np.around(apoptosis_max,3)

print(f"Correlations: \t\tMin\t\tMax")
print("========================================================")
print(f"parthanatos \t\t{parthanatos_min},\t\t{parthanatos_max}")
print(f"necroptosis \t\t{necroptosis_min},\t\t{necroptosis_max}")
print(f"apoptosis \t\t{apoptosis_min},\t\t{apoptosis_max}")

Apoptosis seems to have popped out as the strongest but not by a massive amount. It has (negative) 33% correlation to PC 2

# Population Plot

In [ ]:
# Pairwise proportion plots
data.obs['leiden'].unique()

In [ ]:
from misc_utils import auto_split_range

def get_pairwise_proportions(data,n_leiden=20):

    # sizes = data.obs.groupby('leiden').size()
    sizes = np.array([np.sum(data.obs['leiden'] == str(i)) for i in range(n_leiden)])
    pairwise = np.outer((1+sizes),1/(1+sizes))
    return pairwise

pairwise = get_pairwise_proportions(data)

plt.figure()
plt.imshow(
    np.log(pairwise),
    **auto_split_range(np.log(pairwise))
)
plt.colorbar()
plt.show()

n_clusters = len(data.obs['leiden'].unique())

compound_ratios = {'injury':{},'control':{}}

compound_masks = get_compound_masks(data)
fig,axes = plt.subplots(2,4,figsize=(20,10))
for i,condition in enumerate(compound_masks):
    for j,time in enumerate(compound_masks[condition]):
        ax = axes[i][j]
        subset = data[compound_masks[condition][time]]
        pairwise = get_pairwise_proportions(subset,n_leiden=n_clusters)
        compound_ratios[condition][time] = pairwise
        im = ax.imshow(
            np.log(pairwise),
            **auto_split_range(np.log(pairwise),force_range=2)
        )
        plt.colorbar(im,ax=ax)
        ax.set_xticks(np.arange(n_clusters),labels=np.arange(n_clusters),rotation=90)
        ax.set_yticks(np.arange(n_clusters),labels=np.arange(n_clusters))
        ax.set_title(f"{condition} {time}")
fig.tight_layout()
fig.show()


In [ ]:
fig,axes = plt.subplots(1,4,figsize=(20,5))
fig.suptitle("Control / Injury Ratio of Ratios")
for time,ax in zip(compound_masks['control'],axes):
    ror = (compound_ratios['control'][time])/(compound_ratios['injury'][time])
    im = ax.imshow(
        np.log(ror),
        **auto_split_range(np.log(ror))
    )
    ax.set_xticks(np.arange(n_clusters),labels=np.arange(n_clusters),rotation=90)
    ax.set_yticks(np.arange(n_clusters))
    plt.colorbar(im,ax=ax)

fig.tight_layout()
fig.show()


In [ ]:
for a in compound_ratios['injury'].values():
    print(a.shape)

In [ ]:
# compound_masks = get_compound_masks(data)
# rgc_target_10 = data.obs['leiden'] == "10"
# rgc_target_3 = data.obs['leiden'] == "3"
# rgc_target_both = d

# compound_scores = get_compound_scores(compound_masks,rgc_target_10.astype(dtype=int))
# compound_means = get_compound_means(compound_scores)

target_7 = data.obs['leiden'] == "7"


plt.figure()
plot_timecourse(
    target_7.astype(int),compound_masks,plt.gca(),
    plot_shadow=False,central_tendency="mean", # ax_label=
)
plt.show()

In [ ]:
compound_masks = get_compound_masks(data)
clusters = sorted(list(set(data.obs['leiden'])),key=float)

fig,axes = plt.subplots(4,5,figsize=(15,15))
for cluster,ax in zip(clusters,axes.flatten()):
    cluster_target = data.obs['leiden'] == cluster
    plot_timecourse(
        cluster_target,compound_masks,ax,
        ax_label=cluster,plot_shadow=False
    )
fig.tight_layout()
fig.show()

In [ ]:
rgc_3.shape

In [ ]:
for time in [12,24,48,72]:
    print(f"{time}: {np.sum((rgc_3.obs['time'] == time) & (rgc_3.obs['exp_condition'] == "Cntr"))}")
    

In [ ]:
for time in [12,24,48,72]:
    print(f"{time}: {np.sum((rgc_3.obs['time'] == time) & (rgc_3.obs['exp_condition'] == "Mtz"))}")


In [ ]:
for time in [12,24,48,72]:
    print(f"Injury: {time}, {np.sum((data.obs['time'] == time) & (data.obs['exp_condition'] == "Mtz"))}")
for time in [12,24,48,72]:
    print(f"Control: {time}, {np.sum((data.obs['time'] == time) & (data.obs['exp_condition'] == "Cntr"))}")


In [ ]:
np.sum(data.obs['leiden'] == "19")

In [ ]:
sc.pl.dotplot(data[data.obs['exp_condition'] == "Cntr"],['parthanatos','necroptosis','apoptosis'],groupby="leiden",figsize=(5,10))
sc.pl.dotplot(data[data.obs['exp_condition'] == "Mtz"],['parthanatos','necroptosis','apoptosis'],groupby="leiden",figsize=(5,10))

In [ ]:
# pairwise